In [ ]:
%pip install -q "transformers>=4.51,<5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from pathlib import Path
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Common Setup (Manual Page 3)
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = Path("lab_outputs")
out.mkdir(exist_ok=True)
print("Device:", device)

# Step 1: Load the Model (Manual Page 10)
model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()

# Step 2: Compare Prompts and Decoding (Manual Page 10)
def continue_text(prompt, *, sample=False, temperature=0.8, seed=42):
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    options = {
        "do_sample": sample,
        "max_new_tokens": 45,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if sample:
        options.update(temperature=temperature, top_p=0.9)
    with torch.no_grad():
        tokens = model.generate(**inputs, **options)
    return tokenizer.decode(tokens[0, inputs.input_ids.size(1):], skip_special_tokens=True).strip()

prompts = [
    "A story about a robot begins like this:",
    "In a school library, a tiny robot found a lost book. It",
]

for prompt in prompts:
    print("\nPROMPT:", prompt)
    print("GREEDY:", continue_text(prompt))
    for temp in (0.6, 1.0):
        for seed in (42, 43):
            print(f"Temp={temp}, Seed={seed}:", continue_text(prompt, sample=True, temperature=temp, seed=seed))

Device: cpu


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


PROMPT: A story about a robot begins like this:
GREEDY: A robot is a robot that is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed
Temp=0.6, Seed=42: The robot was a humanoid robot with a heart, eyes, and hands that looked like a human, but its skin was red. The robot had a small, muscular body and was very muscular. It was also very slow.
Temp=0.6, Seed=43: The robot, a robot named "The Robot," is trying to figure out what happened to its owner, who is a woman. The robot tries to figure out how to get back to her, and then it has to figure
Temp=1.0, Seed=42: The Robot Man was brought to Earth by the Galactic government, to escape the dark and deadly dangers of space. The robot is the most powerful of all the robots, and even better. It has a long memory for what was
Temp=1.0, Seed=43: The person in charge of the robots, his robot friend, or even their d

In [ ]:
# Step 3: Fine-Tune on Study Tip Examples (Manual Page 12)
examples = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
]

tuned = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# Freeze the base model and train only the last Transformer block
tuned.requires_grad_(False)
tuned.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned.transformer.h[-1].parameters(), lr=2e-5)

for epoch in range(3):
    tuned.train()
    for topic, answer in examples:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
    print("Epoch:", epoch + 1, "last example loss:", round(loss.item(), 3))

tuned.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")
print("Saved fine-tuned model to lab_outputs/study_tip_gpt2")

# Step 4: Compare on Held-Out Topics (Manual Page 12)
held_out = ["recursion", "matrices", "binary search"]
base_model = model
print("\n--- HELD-OUT EVALUATION ---")
for topic in held_out:
    prompt = f"Topic: {topic}\nStudy tip:"
    model = base_model
    print(f"\n[{topic.upper()}] BASE :", continue_text(prompt))
    model = tuned.eval()
    print(f"[{topic.upper()}] TUNED:", continue_text(prompt))
model = base_model

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch: 1 last example loss: 4.991
Epoch: 2 last example loss: 4.534
Epoch: 3 last example loss: 4.832
Saved fine-tuned model to lab_outputs/study_tip_gpt2

--- HELD-OUT EVALUATION ---

[RECURSION] BASE : Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int i) { return i; } public: Recursion(int i)
[RECURSION] TUNED: Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int x, int y) { return x + y; } public: Rec

[MATRICES] BASE : Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as
[MATRICES] TUNED: Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as

[BINARY SEARCH] BASE : Use the search box to find the search 

In [4]:
import shutil
from google.colab import files

shutil.make_archive("study_tip_gpt2", "zip", "lab_outputs/study_tip_gpt2")
files.download("study_tip_gpt2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>